In [2]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 92.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU 
Model page: https://huggingface.co/mradermacher/ideology_predictor_roberta-GGUF

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/mradermacher/ideology_predictor_roberta-GGUF)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [3]:
!pip install -q gguf
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 3.9 MB/s eta 0:00:00
done


In [4]:

!pip install -q transformers datasets evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [5]:
import os, time, json, random, requests
import numpy as np
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix

set_seed(42)
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

GPU available: True
GPU: Tesla T4


In [6]:
from datasets import load_dataset
import pandas as pd

print("Downloading dataset from Hugging Face...")
hf_ds = load_dataset("Faith1712/Allsides_political_bias_proper", split="train")
df_raw = pd.DataFrame(hf_ds)
print(f"Loaded {len(df_raw)} rows!")
print("\nColumns found:", df_raw.columns.tolist())
print("\nFirst 3 rows:")
display(df_raw.head(3))

README.md: 0.00B [00:00, ?B/s]

allsides_data_unstructured.zip:   0%|          | 0.00/39.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17362 [00:00<?, ? examples/s]

Loaded 17362 rows!

Columns found: ['text', 'label']

First 3 rows:


,text,label
0,"Rep. Elijah Cummings, D-Md., speaks during a l...",1
1,President Barack Obama narrowly won...,1
2,Infrastructure negotiations between President ...,1


In [7]:
from sklearn.model_selection import train_test_split

df = df_raw.copy()
df = df.dropna(subset=['text', 'label']) #drop nulls
df = df[df['text'].str.len() > 50] #drop articles that are too short to have meaningful ideology
df = df.drop_duplicates(subset='text') #drop duplicates

#ensure labels are integers (0=Left, 1=Center, 2=Right)
df['label'] = df['label'].astype(int)

df['text'] = df['text'].str[:1500] #truncate text to 1500 chars

print(f'Clean dataset size: {len(df)}')
print("Label distribution:")
print(df['label'].value_counts())

#split into train (70%), validation (15%), test (15%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print('\nSplit successful:')
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Clean dataset size: 17059
Label distribution:
label
0    7672
2    5399
1    3988
Name: count, dtype: int64

Split successful:
Train: 11941 | Val: 2559 | Test: 2559


In [8]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'roberta-base'
MAX_LEN = 512
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded successfully!')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


In [9]:
class IdeologyDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts     = dataframe['text'].tolist()
        self.labels    = dataframe['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = IdeologyDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = IdeologyDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = IdeologyDataset(test_df,  tokenizer, MAX_LEN)
print('Datasets ready')

Datasets ready


In [10]:
from transformers import AutoModelForSequenceClassification

#labels mapping (0=Left, 1=Center, 2=Right)
id2label = {0: 'Left', 1: 'Center', 2: 'Right'}
label2id = {v: k for k, v in id2label.items()}

print("Downloading RoBERTa model")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

total = sum(p.numel() for p in model.parameters())
print(f'Model loaded successfully! Parameters: {total:,}')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully! Parameters: 124,647,939


In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': float((preds == labels).mean()),
    }

training_args = TrainingArguments(
    output_dir='./roberta_ideology',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type='linear',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
trainer.train()

import shutil
import json

SAVE_DIR = './ideology_model_final'
#save model
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

#save label mapping
with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id}, f)

print("Model, tokenizer, and mappings saved locally")
print("Zipping the model files...")

shutil.make_archive("deberta_ideology_model", 'zip', SAVE_DIR)

print("Success! 'roberta_ideology_model.zip' is ready")

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,2.798943,0.642522,0.878142,0.880422
2,2.265043,0.550977,0.899406,0.901915
3,1.251414,0.611848,0.901579,0.905041
4,0.921038,0.707706,0.904576,0.908167
5,0.766955,0.773678,0.900912,0.904650


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model, tokenizer, and mappings saved locally
Zipping the model files...
Success! 'roberta_ideology_model.zip' is ready


In [12]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

LABEL_NAMES = ['Left', 'Center', 'Right']

#get predictions
preds_output = trainer.predict(test_dataset)
preds  = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

#print results
print('=' * 55)
print('TEST SET RESULTS')
print('=' * 55)
print(classification_report(labels, preds, target_names=LABEL_NAMES))

print('\nConfusion Matrix (rows=true, cols=pred):')
cm = confusion_matrix(labels, preds)
print(pd.DataFrame(
    cm,
    index=[f'True {l}' for l in LABEL_NAMES],
    columns=[f'Pred {l}' for l in LABEL_NAMES],
))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TEST SET RESULTS
              precision    recall  f1-score   support

        Left       0.87      0.95      0.91      1151
      Center       0.90      0.83      0.86       598
       Right       0.94      0.87      0.90       810

    accuracy                           0.90      2559
   macro avg       0.90      0.88      0.89      2559
weighted avg       0.90      0.90      0.90      2559


Confusion Matrix (rows=true, cols=pred):
             Pred Left  Pred Center  Pred Right
True Left         1098           24          29
True Center         87          495          16
True Right          76           29         705


In [13]:
#Per-example probabilities
import torch
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=-1).numpy()

idx = 0
print(f'Text:      {test_df.iloc[idx]["text"][:150]}')
print(f'True:      {LABEL_NAMES[labels[idx]]}')
print(f'Predicted: {LABEL_NAMES[preds[idx]]}')
for i, name in enumerate(LABEL_NAMES):
    print(f'  {name}: {probs[idx][i]:.4f}')

Text:      NEWYou can now listen to Fox News articles!  President Obama said Wednesday that the use of chemical weapons in Syria would be a "game-changer" that w
True:      Right
Predicted: Right
  Left: 0.0006
  Center: 0.0006
  Right: 0.9988


In [14]:
from transformers import pipeline
import torch

ideology_clf = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=512,
)

def predict_ideology(text):
    raw_output = ideology_clf(text)
    
    if isinstance(raw_output[0], list):
        results = raw_output[0]
    else:
        results = raw_output
        
    return {r['label']: round(r['score'], 4) for r in results}

#test model
tests = [
    'The president\'s tax cuts primarily benefit the wealthy while cutting social programs.',
    'The new infrastructure bill passed with bipartisan support in the Senate.',
    'The government\'s immigration crackdown is essential for national security.',
]

print("Running predictions...\n")
for t in tests:
    p = predict_ideology(t)
    print(f'{max(p, key=p.get):6s} | {p}')
    print(f'       {t[:80]}')
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running predictions...

Left   | {'Left': 0.9968, 'Right': 0.0016, 'Center': 0.0016}
       The president's tax cuts primarily benefit the wealthy while cutting social prog

Left   | {'Left': 0.995, 'Center': 0.0029, 'Right': 0.0021}
       The new infrastructure bill passed with bipartisan support in the Senate.

Left   | {'Left': 0.998, 'Center': 0.0012, 'Right': 0.0007}
       The government's immigration crackdown is essential for national security.



In [15]:
import re

def score_sentences(article_text, top_k=5):
    sentences = re.split(r'(?<=[.!?])\s+', article_text.strip())
    sentences = [s for s in sentences if len(s) > 20]
    scored = []
    for sent in sentences:
        probs = predict_ideology(sent)
        best_label = max(probs, key=probs.get)
        scored.append({
            'sentence': sent,
            'predicted': best_label,
            'probs': probs,
            'confidence': probs[best_label],
        })
    scored.sort(key=lambda x: x['confidence'], reverse=True)
    return scored[:top_k]


sample = (
    'The administration unveiled its new economic plan today. '
    'Critics argue that the tax cuts disproportionately benefit corporations. '
    'Supporters say the plan will spur job creation and reduce the deficit. '
    'The proposal includes significant cuts to environmental regulations. '
    'Many independent economists project mixed results in the first year. '
    'The president signed the order calling it a historic achievement.'
)

print('Top ideologically salient sentences:')
print('=' * 60)
for item in score_sentences(sample):
    print(f"[{item['predicted']:6s} | conf={item['confidence']:.3f}] {item['sentence']}")
    print(f"  {item['probs']}")
    print()

Top ideologically salient sentences:
[Left   | conf=0.986] The administration unveiled its new economic plan today.
  {'Left': 0.9858, 'Center': 0.01, 'Right': 0.0042}

[Left   | conf=0.961] Critics argue that the tax cuts disproportionately benefit corporations.
  {'Left': 0.9606, 'Center': 0.0378, 'Right': 0.0016}

[Left   | conf=0.839] Supporters say the plan will spur job creation and reduce the deficit.
  {'Left': 0.8389, 'Center': 0.1558, 'Right': 0.0052}

[Left   | conf=0.784] The proposal includes significant cuts to environmental regulations.
  {'Left': 0.7836, 'Center': 0.1843, 'Right': 0.0321}

[Left   | conf=0.557] Many independent economists project mixed results in the first year.
  {'Left': 0.5575, 'Center': 0.4357, 'Right': 0.0069}



In [19]:
import shap
import numpy as np
import re
import torch

# Initialize the SHAP explainer
explainer = shap.Explainer(ideology_clf)

def analyze_by_shap_ranking(article_text, top_k=5):
    # 1. Get the SHAP values for the whole article
    # This captures how sentences interact with each other
    shap_values = explainer([article_text])
    
    # 2. Determine the article's overall predicted label
    probs = predict_ideology(article_text)
    predicted_label = max(probs, key=probs.get)
    label_idx = ideology_clf.model.config.label2id[predicted_label]
    
    print(f"Overall Article Prediction: {predicted_label}")
    print(f"Ranking sentences by SHAP contribution to '{predicted_label}'...")
    print('=' * 60)

    # 3. Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', article_text.strip())
    sentences = [s for s in sentences if len(s) > 15]
    
    # 4. Map the token-level SHAP values back to sentences
    scored_sentences = []
    tokens = shap_values.data[0]
    token_values = shap_values.values[0, :, label_idx]
    
    current_token_idx = 0
    for sent in sentences:
        sent_score = 0
        combined_text = ""
        
        # Accumulate scores for tokens that belong to this sentence
        while current_token_idx < len(tokens) and len(combined_text) < len(sent):
            sent_score += token_values[current_token_idx]
            combined_text += tokens[current_token_idx]
            current_token_idx += 1
            
        scored_sentences.append({
            'sentence': sent,
            'shap_score': sent_score
        })

    # 5. Rank by absolute SHAP score (Highest magnitude of influence)
    scored_sentences.sort(key=lambda x: abs(x['shap_score']), reverse=True)
    top_ranked = scored_sentences[:top_k]

    # 6. Display results and Impact Plots
    for i, item in enumerate(top_ranked):
        direction = "Positive (Supports)" if item['shap_score'] > 0 else "Negative (Opposes)"
        print(f"\nRANK {i+1} | SHAP Score: {item['shap_score']:.4f} | {direction}")
        print(f"Sentence: {item['sentence']}")
        
        sent_shap = explainer([item['sentence']])
        shap.plots.text(sent_shap[0, :, label_idx])
        print("-" * 60)

sample_article = (
    'The administration unveiled its new economic plan today. '
    'Critics argue that the tax cuts disproportionately benefit corporations. '
    'Supporters say the plan will spur job creation and reduce the deficit. '
    'The proposal includes significant cuts to environmental regulations. '
    'Many independent economists project mixed results in the first year. '
    'The president signed the order calling it a historic achievement.'
)

analyze_by_shap_ranking(sample_article, top_k=5)

  0%|          | 0/498 [00:00<?, ?it/s]

Overall Article Prediction: Left
Ranking sentences by SHAP contribution to 'Left'...

RANK 1 | SHAP Score: 0.2667 | Positive (Supports)
Sentence: Many independent economists project mixed results in the first year.


------------------------------------------------------------

RANK 2 | SHAP Score: 0.1718 | Positive (Supports)
Sentence: Critics argue that the tax cuts disproportionately benefit corporations.


------------------------------------------------------------

RANK 3 | SHAP Score: -0.1027 | Negative (Opposes)
Sentence: The president signed the order calling it a historic achievement.


------------------------------------------------------------

RANK 4 | SHAP Score: -0.0735 | Negative (Opposes)
Sentence: The administration unveiled its new economic plan today.


------------------------------------------------------------

RANK 5 | SHAP Score: 0.0343 | Positive (Supports)
Sentence: Supporters say the plan will spur job creation and reduce the deficit.


------------------------------------------------------------
